In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

In [ ]:
def alimarket_scrape(area: str, pages: int):
    # This function scrapes business data from alimarket.es.
    # Data in particular is scraped from https://www.alimarket.es/buscador_empresas_resultados, based on 'area' specified in the string.
    
    # Inputs:
    # 'area' must correspond to the term in Spanish given on the website, such as 'electro' or 'construccion'; in particular it must be the term in spanish that then is in the url once the filter is applied.
    # 'pages' is number of pages to be scraped based on the area specified. This is based on the default configuration of the site with 15 results per page.

    baseurl = 'https://www.alimarket.es'

    headers = {
        'User-Agent =': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Version/4.0 Chrome/130.0.6723.58 Safari/537.36 (AirWatch Browser v21.09.0.9)'
    }
    company_names = []
    region = []
    activity_sector = []
    company_urls = []

    for pages in tqdm(range(1,pages+1)):
        r = requests.get(f'https://www.alimarket.es/buscador_empresas_resultados/area-{area}/pagina-{pages}')

        soup = BeautifulSoup(r.content)

        tables = soup.findChildren('table')
        my_table = tables[0]
        rows = my_table.findChildren(['tr'])

        for row in rows:
            cells = row.findChildren('td')
            for cell_no, cell in enumerate(cells):
                value = cell.string
                #print("The value in this cell is %s" % value)
                if cell_no%3 == 0:
                    company_names.append(value)
                if cell_no%3 == 1:
                    region.append(value)
                if cell_no%3 == 2:
                    activity_sector.append(value)

        for company in rows:
            for link in company.find_all('a', href=True):
                company_urls.append(baseurl+link['href'])
        #        print(link['href'])
    company_phone = []
    company_address = []

    for url in tqdm(company_urls):
        r = requests.get(url, headers=headers)
        soup = BeautifulSoup(r.content)
        for dt, dd in zip(soup.select('#main-inner dl > dt'),
                        soup.select('#main-inner dl > dd')):
            if 'Dirección' in dt.text.strip():
                address = dd.text.strip()
            if 'Contacto' in dt.text.strip():
                phone = dd.get_text(separator=" ").strip()
                if phone == "Contenido exclusivo para suscriptores o compra":
                    phone = "Phone number not found."

        company_phone.append(phone)
        company_address.append(address)
    data = {'Company Name': company_names, 'Region': region, 'Activity Sector': activity_sector, 'Company Phone': company_phone, 'Company Address': company_address}
    df = pd.DataFrame(data=data)
    df.to_excel(f'alimarket_scraped_{area}.xlsx', index=False)